# Basic Inferencing using RDFS and OWL-RL — User Guide


Inferencing is the process of deriving new information from existing graph information according to a specified set of rules, semantics, or expressions.  Inferencing creates new triples based on existing triples in the graph.  

StarLayer supports inferencing in a variety of areas. 
- rdfs/owl-rl inferencing (this guide)
- shacl rules ([SHACL inference rules](04b-shacl-inference-rules.ipynb))
- sparql rules (pending)
- hermit reasoning (pending)

Owlrl is a python package to implement rdfs/owl-rl reasoning. RDF 1.2's one new term kind, triple terms, is not RDFS/OWL-RL vocabulary itself, so it doesn't change what gets entailed - but it does trip up `owlrl` directly: RDFS's own axioms assert `rdf:type rdfs:Resource` for every term seen anywhere in a triple, including subject position, and a triple term is never legal there under RDF 1.2, so calling `owlrl.DeductiveClosure(...).expand(g)` on a graph containing one crashes outright. `StarLayerGraph.infer()` (section 3 below) handles this for you.

This guide provides a brief overview of using owlrl with StarLayerGraph.

## How to run this notebook

See [Getting Started](01-getting-started.ipynb) if you haven't installed StarLayer yet. This guide assumes StarLayer has been pip installed.

Run cells from top to bottom — later sections may reuse variables from earlier sections.

In [1]:
from starlayer import StarLayerGraph, Namespace, RDF

# owlrl is a required (non-optional) dependency of pyshacl, which starshacl
# requires - always present alongside a standard starlayer install.
import owlrl

EX = Namespace("http://example.org/")


def show(graph, label="Resulting graph"):
    """Serialize a graph as Turtle 1.2 with a label - used throughout this
    guide to show what a closure actually added.

    Filtered to ex: subjects only: DeductiveClosure.expand() also adds
    a series of RDFS/OWL/XSD axiomatic vocabulary triples (rdfs:Resource typing,
    owl:sameAs reflexivity for every built-in class/datatype, etc.) that
    would add unneeded complexity to the example output.

    run print(g.serialize(format="turtle12")) to see complete graph.
    """
    filtered = StarLayerGraph()
    filtered.bind("ex", EX)
    for s, p, o in graph:
        if str(s).startswith(str(EX)):
            filtered.add((s, p, o))
    print(f"\n{label}")
    print(filtered.serialize(format="turtle12"))

## 1. RDFS reasoning

`owlrl.DeductiveClosure(owlrl.RDFS_Semantics).expand(graph)` materializes RDFS entailments directly into the graph, , including `rdfs:subClassOf` transitivity as show below.  



In [2]:
# alice is a manager which means she is also an exmployee
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:alice a ex:Manager .
""", format="turtle")

show(g, "Before closure")

owlrl.DeductiveClosure(owlrl.RDFS_Semantics).expand(g)

show(g, "After closure")




Before closure
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Manager rdfs:subClassOf ex:Employee .

ex:alice a ex:Manager .


After closure
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Employee a rdfs:Resource .

ex:Manager a rdfs:Resource ;
    rdfs:subClassOf ex:Employee .

ex:alice a ex:Employee, ex:Manager, rdfs:Resource .



## 2. OWL 2 RL reasoning

`owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(graph)` runs the same kind of forward-chaining closure, but over the OWL 2 RL rule set — entailments RDFS alone can't produce, such as `owl:equivalentClass` and property characteristics like `owl:TransitiveProperty`.

### 2.1 `owl:equivalentClass`

Two systems label the same role differently — HR calls it `ex:Manager`, the CRM calls the same thing `ex:TeamLead`. Declaring them `owl:equivalentClass` lets an instance of one be recognized as the other.

In [3]:
# alice is a manager, but she is also a teamlead, because manager and teamlead at the same
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Manager owl:equivalentClass ex:TeamLead .
    ex:alice a ex:Manager .
""", format="turtle")

show(g, "Before closure")

owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(g)

show(g, "After closure")


Before closure
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:Manager owl:equivalentClass ex:TeamLead .

ex:alice a ex:Manager .


After closure
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Manager rdfs:subClassOf ex:TeamLead ;
    owl:equivalentClass ex:TeamLead ;
    owl:sameAs ex:Manager .

ex:TeamLead rdfs:subClassOf ex:Manager ;
    owl:equivalentClass ex:Manager ;
    owl:sameAs ex:TeamLead .

ex:alice a ex:TeamLead, ex:Manager ;
    owl:sameAs ex:alice .



### 2.2 `owl:TransitiveProperty`

Declaring `ex:partOf` to be an `owl:TransitiveProperty` lets a chain of `partOf` facts entail the full transitive closure, not just each direct link.

In [4]:
# the SalesTeam is part of the SalesDepartment, which means the SalesTeam is a part of AcmeCorp
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:partOf a owl:TransitiveProperty .
    ex:SalesTeam ex:partOf ex:SalesDept .
    ex:SalesDept ex:partOf ex:AcmeCorp .
""", format="turtle")

show(g, "Before closure")

owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(g)

show(g, "After closure")


Before closure
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:SalesDept ex:partOf ex:AcmeCorp .

ex:SalesTeam ex:partOf ex:SalesDept .

ex:partOf a owl:TransitiveProperty .


After closure
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:AcmeCorp owl:sameAs ex:AcmeCorp .

ex:SalesDept ex:partOf ex:AcmeCorp ;
    owl:sameAs ex:SalesDept .

ex:SalesTeam ex:partOf ex:AcmeCorp, ex:SalesDept ;
    owl:sameAs ex:SalesTeam .

ex:partOf a owl:TransitiveProperty ;
    owl:sameAs ex:partOf .



### 2.3 Running RDFS and OWL 2 RL together

`owlrl.RDFS_OWLRL_Semantics` combines both rule sets into a single closure — one `expand()` call produces both RDFS entailments (like section 1's `rdfs:subClassOf`) and OWL 2 RL entailments (like 2.1's `owl:equivalentClass` and 2.2's `owl:TransitiveProperty`) at once, not one after the other.

In [5]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .

    ex:Manager rdfs:subClassOf ex:Employee .
    ex:Manager owl:equivalentClass ex:TeamLead .
    ex:alice a ex:Manager .

    ex:partOf a owl:TransitiveProperty .
    ex:SalesTeam ex:partOf ex:SalesDept .
    ex:SalesDept ex:partOf ex:AcmeCorp .
""", format="turtle")

show(g, "Before closure")

owlrl.DeductiveClosure(owlrl.RDFS_OWLRL_Semantics).expand(g)

show(g, "After closure")


Before closure
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Manager rdfs:subClassOf ex:Employee ;
    owl:equivalentClass ex:TeamLead .

ex:SalesDept ex:partOf ex:AcmeCorp .

ex:SalesTeam ex:partOf ex:SalesDept .

ex:alice a ex:Manager .

ex:partOf a owl:TransitiveProperty .


After closure
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:AcmeCorp a owl:Thing, rdfs:Resource ;
    owl:sameAs ex:AcmeCorp .

ex:Employee a owl:Thing, rdfs:Resource ;
    owl:sameAs ex:Employee .

ex:Manager a owl:Thing, rdfs:Resource ;
    rdfs:subClassOf ex:TeamLead, ex:Manager, ex:Employee ;
    owl:equivalentClass ex:Manager, ex:TeamLead ;
    owl:sameAs ex:Manager .

ex:SalesDept ex:partOf ex:AcmeCorp ;
    a rdfs:Resource, owl:Thing ;
    owl:sameAs e

## 3. `StarLayerGraph.infer()` — the built-in API

Everything above calls `owlrl` directly. `StarLayerGraph.infer(profile="rdfs"|"owl-rl")` wraps that same workflow as a real method, adding two things the manual pattern above doesn't give you for free:

- **Never mutates the graph it's called on.** `infer()` always materializes into a *new* graph (or a caller-supplied `target_graph`) — the graph you called it on is untouched, unlike `owlrl.DeductiveClosure(...).expand(g)` above, which mutates `g` in place.
- **Triple-term aware.** As the intro noted, `owlrl`'s RDFS closure crashes outright on any graph containing a triple term. `infer()` decomposes triple terms into their synthetic reification form first (the same one `rdfc10_hash()`/`isomorphic()` use — see the [canonical hashing guide](05d-canonical-hashing.ipynb)), so it works on RDF 1.2 data the raw `owlrl` calls above cannot handle at all.

Requires the optional `owlrl` dependency: `pip install starlayergraph[reasoning]`.

Same caveat as everywhere above: no truth maintenance. Retracting a fact from the original graph later does not retract anything a previous `infer()` call entailed from it — call `infer()` again from the original facts after any edit, rather than trying to patch a previous call's output.

In [6]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:alice a ex:Manager .
""", format="turtle")

closed = g.infer(profile="rdfs")
print("original graph untouched:", len(g), "triples")
show(closed, "infer() result")

original graph untouched: 2 triples

infer() result
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Employee a rdfs:Resource .

ex:Manager a rdfs:Resource ;
    rdfs:subClassOf ex:Employee .

ex:alice a ex:Employee, ex:Manager, rdfs:Resource .



### 3.1 Working with triple terms

The same graph as above, plus one RDF 1.2 triple term. `owlrl.DeductiveClosure(owlrl.RDFS_Semantics).expand(g)` on this graph would raise `ValueError: RDF 1.2: triple terms are not permitted in subject position of a triple.` — `infer()` handles it.

In [7]:
from rdflib import URIRef
from starlayer import TripleTerm

REIFIES = URIRef("http://www.w3.org/1999/02/22-rdf-syntax-ns#reifies")

g2 = StarLayerGraph()
g2.bind("ex", EX)
g2.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:alice a ex:Manager .
""", format="turtle")
g2.add((EX.claim, REIFIES, TripleTerm(EX.bob, EX.reportsTo, EX.alice)))

closed2 = g2.infer(profile="rdfs")
print("alice is an Employee:", (EX.alice, RDF.type, EX.Employee) in closed2)

alice is an Employee: True


## Where to go next

1. **[Getting Started](01-getting-started.ipynb)**
2. **[Graphs](02-graphs.ipynb)**
   - 2.b **Inferencing** — this guide.
3. **[SPARQL](03-sparql.ipynb)**
   - 3.a **[SPARQL rules (pending)](03a-sparql-rules-pending.md)** — SPARQL-RL (SRL), a separate Datalog-style rules language.
   - 3.b **[SPARQL inferencing](03b-sparql-inferencing.ipynb)** — `.query(..., entailment="rdfs"|"owl-rl")`, a per-call choice between query-time rdfs:subClassOf rewrite and materialize-query-discard for full RDFS/OWL-RL - the "owl-rl" case uses this guide's own `infer()` internally, fresh for each call, never cached.
4. **[SHACL shapes](04-shacl-shapes.ipynb)**
   - 4.b **[SHACL inference rules](04b-shacl-inference-rules.ipynb)** — SHACL rules including execution ordering, rule sets, provenance.
5. **Other**
   - 5.e Owl reasoning with Hermit (pending) — full OWL DL reasoning; not yet part of this stack.